<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/10-engineering-practices/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 10: Solutions

Try the exercises first. The notes after each solution explain the design choices and the failure each check is designed to catch.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import contextlib
import hashlib
import io
import json
import logging
import math
import platform
import re
import sys
import tempfile
import time
import tracemalloc
from datetime import datetime, timezone
from pathlib import Path

import numpy as np


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _tmpdir():
    return Path(tempfile.mkdtemp())


def check_parse_requirements(f):
    text = """# core
numpy>=1.26
pandas == 2.2.3   # pinned for the paper
Scikit_Learn

-r dev-requirements.txt
uvicorn[standard]>=0.30 ; python_version >= "3.10"
torch!=2.1.0,<3
"""
    expected = {"numpy": ">=1.26", "pandas": "==2.2.3", "scikit-learn": None,
                "uvicorn": ">=0.30", "torch": "!=2.1.0,<3"}
    _scenarios("parse_requirements", [
        ("a typical file", lambda: f(text), expected,
         "strip comments and markers first, skip lines starting with '-', normalise names"),
        ("empty text", lambda: f(""), {}, None),
        ("only comments and blanks", lambda: f("# nothing\n\n   \n"), {}, None),
        ("compatible release ~=", lambda: f("matplotlib~=3.8"), {"matplotlib": "~=3.8"}, None),
        ("duplicate package raises", lambda: f("numpy>=1.26\nNumPy==2.0"), Raises(ValueError),
         "normalise the name before checking for duplicates"),
    ])


def check_make_config(f):
    def defaults_unchanged():
        f({"lr": 0.5, "epochs": 3})
        return DEFAULTS == {"lr": 0.1, "epochs": 100, "batch_size": 32, "seed": 0}

    def overrides_unchanged():
        o = {"lr": 1}
        f(o)
        return o

    _scenarios("make_config", [
        ("no overrides gives the defaults", lambda: f(), {"lr": 0.1, "epochs": 100, "batch_size": 32, "seed": 0}, None),
        ("overrides are applied", lambda: f({"epochs": 5, "seed": 3}), {"lr": 0.1, "epochs": 5, "batch_size": 32, "seed": 3}, None),
        ("an integer lr is stored as a float", lambda: type(f({"lr": 1})["lr"]), float, "config['lr'] = float(lr)"),
        ("a typo in a key raises KeyError", lambda: f({"learning_rate": 0.5}), Raises(KeyError),
         "compare the override keys with DEFAULTS"),
        ("negative lr raises", lambda: f({"lr": -0.1}), Raises(ValueError), None),
        ("lr as a string raises", lambda: f({"lr": "0.1"}), Raises(ValueError), None),
        ("epochs = 0 raises", lambda: f({"epochs": 0}), Raises(ValueError), None),
        ("epochs as a float raises", lambda: f({"epochs": 10.0}), Raises(ValueError), None),
        ("epochs = True raises", lambda: f({"epochs": True}), Raises(ValueError),
         "isinstance(True, int) is True: exclude bools explicitly"),
        ("negative seed raises", lambda: f({"seed": -1}), Raises(ValueError), None),
        ("DEFAULTS is not modified", defaults_unchanged, True, "build a new dict: {**DEFAULTS, **overrides}"),
        ("the overrides dict is not modified", overrides_unchanged, {"lr": 1}, None),
    ])


def check_setup_logger(f):
    def capture(name, calls=1, level="INFO", messages=(("info", "hello"),)):
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            for _ in range(calls):
                log = f(name, level=level)
            for method, text in messages:
                getattr(log, method)(text)
        return buf.getvalue()

    def with_file():
        path = _tmpdir() / "run.log"
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            log = f("check.file", log_file=path)
            log.info("saved")
            n_handlers = len(log.handlers)
        for h in log.handlers:
            h.flush()
        return (n_handlers, path.read_text())

    _scenarios("setup_logger", [
        ("returns a Logger", lambda: isinstance(f("check.type"), logging.Logger), True, None),
        ("format is 'LEVEL name: message'", lambda: capture("check.fmt"), "INFO check.fmt: hello\n",
         "logging.Formatter('%(levelname)s %(name)s: %(message)s') on a StreamHandler(sys.stdout)"),
        ("calling it three times doesn't repeat messages", lambda: capture("check.dup", calls=3),
         "INFO check.dup: hello\n", "remove existing handlers before adding new ones"),
        ("one handler after repeated calls", lambda: [len(f("check.count").handlers) for _ in range(3)], [1, 1, 1], None),
        ("the level hides lower messages", lambda: capture("check.level", level="WARNING",
                                                           messages=[("info", "quiet"), ("warning", "loud")]),
         "WARNING check.level: loud\n", "logger.setLevel(getattr(logging, level.upper()))"),
        ("doesn't propagate to the root logger", lambda: f("check.prop").propagate, False, None),
        ("log_file adds a second handler that writes the same format", with_file,
         (2, "INFO check.file: saved\n"), "add a logging.FileHandler(log_file) with the same formatter"),
    ])


def check_file_sha256(f):
    folder = _tmpdir()
    small = folder / "small.txt"
    small.write_bytes(b"penguins\n")
    big = folder / "big.bin"
    big.write_bytes(bytes(range(256)) * 1000)
    empty = folder / "empty.bin"
    empty.write_bytes(b"")
    _scenarios("file_sha256", [
        ("a small file", lambda: f(small), hashlib.sha256(b"penguins\n").hexdigest(), None),
        ("chunks give the same hash as reading at once", lambda: f(big, chunk_size=7),
         hashlib.sha256(big.read_bytes()).hexdigest(), "digest.update(chunk) for each chunk"),
        ("an empty file", lambda: f(empty), hashlib.sha256(b"").hexdigest(), None),
        ("a path given as a string", lambda: f(str(small)), hashlib.sha256(b"penguins\n").hexdigest(), None),
        ("returns 64 hex characters", lambda: len(f(small)), 64, "return digest.hexdigest()"),
    ])


def check_Timer(f):
    def basic():
        with f() as t:
            time.sleep(0.05)
        return 0.04 < t.elapsed < 1.0

    def before():
        return f().elapsed

    def enter_returns_self():
        timer = f()
        with timer as t:
            pass
        return t is timer

    def with_error():
        timer = f()
        try:
            with timer:
                time.sleep(0.02)
                raise KeyError("boom")
        except KeyError:
            return timer.elapsed is not None and timer.elapsed > 0.01
        return "the exception was swallowed"

    def reuse():
        timer = f()
        with timer:
            time.sleep(0.05)
        with timer:
            pass
        return timer.elapsed < 0.04

    def logs():
        log = logging.getLogger("check.timer")
        log.handlers.clear()
        log.propagate = False
        log.setLevel(logging.INFO)
        buf = io.StringIO()
        handler = logging.StreamHandler(buf)
        log.addHandler(handler)
        with f("loading", logger=log):
            pass
        log.removeHandler(handler)
        return bool(re.fullmatch(r"loading took \d+\.\d{3} s\n", buf.getvalue()))

    _scenarios("Timer", [
        ("elapsed is None before use", before, None, None),
        ("measures a 0.05 s sleep", basic, True, "time.perf_counter() in __enter__ and __exit__"),
        ("__enter__ returns the timer", enter_returns_self, True, "return self from __enter__"),
        ("records the time and lets the exception through", with_error, True,
         "__exit__ must return False (or None), never True"),
        ("each use measures afresh", reuse, True, "restart the clock in __enter__"),
        ("logs '<label> took <seconds> s'", logs, True, 'logger.info("%s took %.3f s", label, elapsed)'),
    ])


def check_retry(f):
    def flaky(fail_times, exc=ConnectionError):
        calls = []

        def download():
            """Fetch something."""
            calls.append(1)
            if len(calls) <= fail_times:
                raise exc("network down")
            return "data"
        return download, calls

    def run(fail_times, times=3, exceptions=(ConnectionError,), exc=ConnectionError):
        func, calls = flaky(fail_times, exc)
        try:
            result = f(times=times, exceptions=exceptions)(func)()
        except Exception as e:
            result = type(e).__name__
        return (result, len(calls))

    def sleeps_with_delay():
        recorded = []
        real_sleep = time.sleep
        time.sleep = recorded.append
        try:
            func, _ = flaky(5)
            try:
                f(times=3, exceptions=(ConnectionError,), delay=0.5)(func)()
            except ConnectionError:
                pass
        finally:
            time.sleep = real_sleep
        return recorded

    def keeps_name():
        func, _ = flaky(0)
        wrapped = f()(func)
        return (wrapped.__name__, wrapped.__doc__)

    _scenarios("retry", [
        ("succeeds first time: one call", lambda: run(0), ("data", 1), None),
        ("fails twice then succeeds", lambda: run(2), ("data", 3), None),
        ("always fails: re-raises after 3 calls", lambda: run(10), ("ConnectionError", 3), "after the last attempt, raise"),
        ("other exceptions are not retried", lambda: run(10, exc=KeyError), ("KeyError", 1),
         "only catch the exception types you were given"),
        ("times=1 means no retries", lambda: run(10, times=1), ("ConnectionError", 1), None),
        ("sleeps between attempts, not after the last", sleeps_with_delay, [0.5, 0.5],
         "3 attempts have 2 gaps between them"),
        ("keeps the function's name and docstring", keeps_name, ("download", "Fetch something."), "@functools.wraps(func)"),
        ("times=0 raises", lambda: f(times=0), Raises(ValueError), None),
    ])


def check_split_indices(f):
    def covers():
        train, test = f(100, 0.2, seed=1)
        return (len(train), len(test), sorted(np.concatenate([train, test]).tolist()) == list(range(100)))

    def global_state():
        state = np.random.get_state()[1].copy()
        f(50, 0.3, seed=7)
        return bool(np.array_equal(state, np.random.get_state()[1]))

    def same_seed():
        a, b = f(30, 0.3, seed=5), f(30, 0.3, seed=5)
        return bool(np.array_equal(a[0], b[0]) and np.array_equal(a[1], b[1]))

    def different_seeds():
        return not np.array_equal(f(30, 0.3, seed=1)[1], f(30, 0.3, seed=2)[1])

    def sorted_ints():
        train, test = f(40, 0.25, seed=0)
        return (bool(np.all(np.diff(train) > 0) and np.all(np.diff(test) > 0)), train.dtype.kind, test.dtype.kind)

    _scenarios("split_indices", [
        ("sizes, and every index exactly once", covers, (80, 20, True), "one permutation, then slice it"),
        ("rounds the test size", lambda: len(f(10, 0.33, seed=0)[1]), 3, None),
        ("sorted integer arrays", sorted_ints, (True, "i", "i"), None),
        ("same seed, same split", same_seed, True, None),
        ("different seeds, different splits", different_seeds, True, None),
        ("doesn't touch NumPy's global random state", global_state, True,
         "use rng = np.random.default_rng(seed), not np.random.seed"),
        ("test_fraction of 1 raises", lambda: f(10, 1.0, seed=0), Raises(ValueError), None),
        ("an empty test set raises", lambda: f(10, 0.01, seed=0), Raises(ValueError), None),
    ])


def _std_correct(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be 2D")
    std = X.std(axis=0)
    return (X - X.mean(axis=0)) / np.where(std == 0, 1.0, std)


def _std_global(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be 2D")
    s = X.std()
    return (X - X.mean()) / (s if s else 1.0)


def _std_ddof1(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be 2D")
    std = X.std(axis=0, ddof=1)
    return (X - X.mean(axis=0)) / np.where(std == 0, 1.0, std)


def _std_inplace(X):
    if np.ndim(X) != 2:
        raise ValueError("X must be 2D")
    std = X.std(axis=0)
    X -= X.mean(axis=0)
    X /= np.where(std == 0, 1.0, std)
    return X


def _std_nan(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be 2D")
    with np.errstate(divide="ignore", invalid="ignore"):
        return (X - X.mean(axis=0)) / X.std(axis=0)


def _std_any_shape(X):
    X = np.asarray(X, dtype=float)
    std = X.std(axis=0)
    return (X - X.mean(axis=0)) / np.where(std == 0, 1.0, std)


def check_test_standardize(f):
    def outcome(impl):
        try:
            result = f(impl)
        except KeyboardInterrupt:
            raise
        except BaseException:          # pytest's failures derive from BaseException, not Exception
            return "failed"
        return "passed" if result is None else f"returned {result!r}"

    _scenarios("test_standardize", [
        ("your test passes the correct version", lambda: outcome(_std_correct), "passed",
         "your test fails on correct code: check your expected values and tolerances"),
        ("catches: one mean and std for the whole array", lambda: outcome(_std_global), "failed",
         "use test data whose columns have very different means and spreads"),
        ("catches: ddof=1 instead of ddof=0", lambda: outcome(_std_ddof1), "failed",
         "check the std of each column is 1 with a tight tolerance, e.g. pytest.approx(1)"),
        ("catches: the input is modified", lambda: outcome(_std_inplace), "failed",
         "take a copy before the call and compare afterwards"),
        ("catches: a constant column gives NaN", lambda: outcome(_std_nan), "failed",
         "test a column where every value is the same"),
        ("catches: non-2D input is accepted", lambda: outcome(_std_any_shape), "failed",
         "check that 1D input raises ValueError"),
    ])


def check_predict_in_batches(f):
    X = np.arange(23 * 3, dtype=float).reshape(23, 3)

    def row_sums(A):
        return A.sum(axis=1)

    def two_cols(A):
        return np.column_stack([A[:, 0], A[:, 0] * 2])

    def batch_sizes():
        seen = []

        def record(A):
            seen.append(len(A))
            return A[:, 0]
        f(record, X, 5)
        return seen

    _scenarios("predict_in_batches", [
        ("1D output matches predicting all at once", lambda: f(row_sums, X, 5), row_sums(X),
         "np.concatenate(outputs, axis=0)"),
        ("2D output matches predicting all at once", lambda: f(two_cols, X, 4), two_cols(X), None),
        ("batches of at most batch_size rows", batch_sizes, [5, 5, 5, 5, 3], "range(0, len(X), batch_size)"),
        ("batch_size bigger than the data", lambda: f(row_sums, X, 100), row_sums(X), None),
        ("batch_size of 1", lambda: f(row_sums, X, 1), row_sums(X), None),
        ("batch_size 0 raises", lambda: f(row_sums, X, 0), Raises(ValueError), None),
    ])


def check_pairwise_distances(f):
    rng = np.random.default_rng(3)
    A, B = rng.normal(size=(7, 4)), rng.normal(size=(5, 4))
    reference = np.sqrt(((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=2))

    def self_distances():
        P = rng.normal(loc=1000, size=(50, 3))
        D = f(P, P)
        return (bool(np.all(np.isfinite(D))), bool(np.all(D >= 0)), bool(np.allclose(np.diag(D), 0, atol=1e-4)))

    def memory():
        X, Y = rng.normal(size=(600, 64)), rng.normal(size=(500, 64))
        tracemalloc.start()
        f(X, Y)
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        return peak / 1e6 < 30

    _scenarios("pairwise_distances", [
        ("matches the direct formula", lambda: f(A, B), reference, None),
        ("shape (n, m)", lambda: f(A, B).shape, (7, 5), None),
        ("integer input", lambda: f([[0, 0], [3, 4]], [[0, 0]]), np.array([[0.0], [5.0]]), None),
        ("distances to itself: no NaN, never negative, diagonal close to 0", self_distances, (True, True, True),
         "clip the squared distances at 0 before np.sqrt"),
        ("peak memory under 30 MB for 600 x 500 points in 64 dimensions", memory, True,
         "expand the square: sum(X**2) + sum(Y**2) - 2 * X @ Y.T, no (n, m, d) array"),
    ])


def check_LRUCache(f):
    def basic():
        c = f(2)
        c.put("a", 1)
        c.put("b", 2)
        return (c.get("a"), c.get("b"), c.get("zzz"), c.get("zzz", "none"), len(c))

    def evicts_oldest():
        c = f(2)
        c.put("a", 1)
        c.put("b", 2)
        c.put("c", 3)
        return ("a" in c, "b" in c, "c" in c, len(c))

    def get_counts_as_use():
        c = f(2)
        c.put("a", 1)
        c.put("b", 2)
        c.get("a")
        c.put("c", 3)
        return ("a" in c, "b" in c, "c" in c)

    def put_updates():
        c = f(2)
        c.put("a", 1)
        c.put("b", 2)
        c.put("a", 10)
        c.put("c", 3)
        return (c.get("a"), "b" in c, len(c))

    def contains_is_not_use():
        c = f(2)
        c.put("a", 1)
        c.put("b", 2)
        _ = "a" in c
        c.put("c", 3)
        return ("a" in c, "b" in c)

    def stats():
        c = f(3)
        c.put("x", None)
        results = [c.get("x"), c.get("y"), c.get("x"), c.get("x", "default")]
        return (results, c.hits, c.misses)

    _scenarios("LRUCache", [
        ("put, get, default and len", basic, (1, 2, None, "none", 2), None),
        ("a full cache evicts the least recently used", evicts_oldest, (False, True, True, 2),
         "OrderedDict.popitem(last=False) removes the oldest"),
        ("get counts as a use", get_counts_as_use, (True, False, True), "move_to_end on every get"),
        ("putting an existing key updates it and counts as a use", put_updates, (10, False, 2), None),
        ("'in' does not count as a use", contains_is_not_use, (False, True), None),
        ("hits and misses, even for a stored None", stats, ([None, None, None, None], 3, 1),
         "check 'key in items', not whether the value is None"),
        ("capacity 0 raises", lambda: f(0), Raises(ValueError), None),
    ])


def check_save_run(f):
    def basic():
        folder = _tmpdir() / "runs" / "exp1"
        path = f({"lr": 0.1}, {"accuracy": 0.9}, folder)
        data = json.loads(Path(path).read_text())
        return (Path(path) == folder / "run.json", sorted(data), data["config"], data["metrics"])

    def environment():
        data = json.loads(Path(f({}, {}, _tmpdir())).read_text())
        return data["environment"] == {"python": platform.python_version(), "numpy": np.__version__}

    def timestamp():
        data = json.loads(Path(f({}, {}, _tmpdir())).read_text())
        created = datetime.fromisoformat(data["created"])
        return created.tzinfo is not None and abs((datetime.now(timezone.utc) - created).total_seconds()) < 60

    def numpy_values():
        metrics = {"accuracy": np.float64(0.91), "n": np.int64(81), "ok": np.bool_(True),
                   "per_class": np.array([0.9, 0.95])}
        data = json.loads(Path(f({}, metrics, _tmpdir())).read_text())
        return data["metrics"]

    def refuses_overwrite():
        folder = _tmpdir()
        f({"v": 1}, {}, folder)
        try:
            f({"v": 2}, {}, folder)
        except FileExistsError:
            return json.loads((folder / "run.json").read_text())["config"]
        return "no error: the first result was replaced"

    def overwrite_allowed():
        folder = _tmpdir()
        f({"v": 1}, {}, folder)
        f({"v": 2}, {}, folder, overwrite=True)
        return json.loads((folder / "run.json").read_text())["config"]

    def bad_value():
        folder = _tmpdir() / "bad"
        try:
            f({}, {"seen": {1, 2}}, folder)
        except TypeError:
            return sorted(p.name for p in folder.glob("*")) if folder.exists() else []
        return "no TypeError raised"

    _scenarios("save_run", [
        ("creates folders and writes the four keys", basic,
         (True, ["config", "created", "environment", "metrics"], {"lr": 0.1}, {"accuracy": 0.9}), None),
        ("records Python and NumPy versions", environment, True, None),
        ("'created' is a time-zone-aware ISO timestamp", timestamp, True, "datetime.now(timezone.utc).isoformat()"),
        ("NumPy numbers and arrays are converted", numpy_values,
         {"accuracy": 0.91, "n": 81, "ok": True, "per_class": [0.9, 0.95]},
         "json.dumps(..., default=converter): .item() for np.generic, .tolist() for arrays"),
        ("refuses to replace an existing run", refuses_overwrite, {"v": 1}, "raise FileExistsError if run.json exists"),
        ("overwrite=True replaces it", overwrite_allowed, {"v": 2}, None),
        ("unsaveable value: TypeError and no file left behind", bad_value, [],
         "build the JSON text before creating any file"),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `parse_requirements`

In [2]:
import re


def parse_requirements(text: str) -> dict:
    requirements = {}
    for line in text.splitlines():
        line = line.split("#", 1)[0].split(";", 1)[0].strip()
        if not line or line.startswith("-"):
            continue
        match = re.match(r"([A-Za-z0-9._-]+)\s*(\[[^\]]*\])?\s*(.*)", line)
        name = match.group(1).lower().replace("_", "-")
        spec = match.group(3).replace(" ", "") or None
        if name in requirements:
            raise ValueError(f"{name} is listed more than once")
        requirements[name] = spec
    return requirements

In [3]:
check_parse_requirements(parse_requirements)

PASS  all 5 tests passed for parse_requirements


The order of the cleaning steps matters. Removing the comment first means a `#` inside a marker or after a version can't confuse anything later. Removing the marker before matching means `python_version >= "3.10"` is never mistaken for part of the version.

The regular expression has three groups: the name (letters, digits, `.`, `_` and `-`), optional extras in brackets, and the rest. `(.*)` takes everything after the name, so `== 2.2.3` and `==2.2.3` both work once spaces are removed.

Normalising names is what pip does too: `Scikit_Learn`, `scikit-learn` and `scikit_learn` are the same package. Without it, the duplicate check would miss `numpy` listed once as `NumPy` and once as `numpy`, with two different versions, which is exactly the kind of conflict you want to catch.

A real tool would use the `packaging` library (`packaging.requirements.Requirement`), which handles every corner of the format. Writing a simple version shows what the format contains.

### Exercise 2: `make_config`

In [4]:
DEFAULTS = {"lr": 0.1, "epochs": 100, "batch_size": 32, "seed": 0}


def _is_int(value) -> bool:
    return isinstance(value, int) and not isinstance(value, bool)


def make_config(overrides=None) -> dict:
    overrides = overrides or {}
    unknown = set(overrides) - set(DEFAULTS)
    if unknown:
        raise KeyError(f"unknown config keys: {sorted(unknown)}")
    config = {**DEFAULTS, **overrides}

    lr = config["lr"]
    if isinstance(lr, bool) or not isinstance(lr, (int, float)) or lr <= 0:
        raise ValueError(f"lr must be a positive number, got {lr!r}")
    config["lr"] = float(lr)
    for key, minimum in [("epochs", 1), ("batch_size", 1), ("seed", 0)]:
        if not _is_int(config[key]) or config[key] < minimum:
            raise ValueError(f"{key} must be an integer of at least {minimum}, got {config[key]!r}")
    return config

In [5]:
check_make_config(make_config)

PASS  all 12 tests passed for make_config


`{**DEFAULTS, **overrides}` builds a new dict: the defaults first, then the overrides on top. Neither input is changed, which matters because `DEFAULTS` is shared by every call. If one call modified it, the next experiment would quietly start from different defaults.

The values are checked **after** merging, so the defaults are checked too. That catches a bad default as early as a bad override.

`{value!r}` in the message shows the value's `repr`, so `"10"` (a string from a badly written config file) appears with its quotes and is easy to tell apart from `10`.

The bool exception looks pedantic, but `True` sneaks in through YAML files, where `epochs: yes` is read as `True`. With a plain `isinstance(x, int)` check, that would train for exactly 1 epoch, because `True == 1`.

### Exercise 3: `setup_logger`

In [6]:
import logging
import sys


def setup_logger(name: str, level: str = "INFO", log_file=None) -> logging.Logger:
    logger = logging.getLogger(name)
    for handler in list(logger.handlers):
        logger.removeHandler(handler)
        handler.close()
    logger.setLevel(getattr(logging, level.upper()))
    logger.propagate = False

    formatter = logging.Formatter("%(levelname)s %(name)s: %(message)s")
    handlers = [logging.StreamHandler(sys.stdout)]
    if log_file is not None:
        handlers.append(logging.FileHandler(log_file))
    for handler in handlers:
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    return logger

In [7]:
check_setup_logger(setup_logger)

PASS  all 7 tests passed for setup_logger


`logging.getLogger(name)` always returns the **same** logger object for the same name, for the life of the Python process. That's why the duplicate problem exists: a notebook cell that adds a handler, run three times, gives a logger with three handlers, and every message is printed three times.

Removing the old handlers makes the function safe to call repeatedly, and `handler.close()` releases any open file. Note the `list(...)`: removing items from a list while looping over it skips some of them, so loop over a copy.

`getattr(logging, "INFO")` turns the level name into the number logging uses internally (20). `logging.getLevelName` can do it too, but its behaviour has changed between Python versions.

In a larger application you'd usually configure logging once, at startup, with `logging.config.dictConfig`, but the idea is the same: one place decides handlers and formats.

### Exercise 4: `file_sha256`

In [8]:
import hashlib


def file_sha256(path, chunk_size: int = 1_048_576) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

In [9]:
check_file_sha256(file_sha256)

PASS  all 5 tests passed for file_sha256


Hashing in chunks gives exactly the same result as hashing the whole file at once: `update(a)` then `update(b)` is the same as `update(a + b)`. That property is what makes it possible to fingerprint a 50 GB dataset with a few megabytes of memory.

`while chunk := f.read(chunk_size):` uses the "walrus" operator (Python 3.8+), which assigns and tests in one step. `read` returns an empty bytes object at the end of the file, which is falsy, so the loop stops.

The file is opened in **binary** mode (`"rb"`). In text mode, Python would decode the bytes and might translate line endings, so the same file could hash differently on Windows and Linux.

Data versioning tools like DVC are built on this idea: store a hash of each data file in git instead of the file itself, and the hash says exactly which version of the data a result came from.

## Medium

### Exercise 5: `Timer`

In [10]:
import time


class Timer:
    def __init__(self, label=None, logger=None):
        self.label = label
        self.logger = logger
        self.elapsed = None
        self._start = None

    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.elapsed = time.perf_counter() - self._start
        if self.logger is not None:
            self.logger.info("%s took %.3f s", self.label or "block", self.elapsed)
        return False                  # don't suppress exceptions

In [11]:
check_Timer(Timer)

PASS  all 6 tests passed for Timer


`with` calls `__enter__` at the start of the block and `__exit__` at the end, **always**, even when the block raises. That's what makes context managers right for anything that must happen afterwards: closing files, releasing locks, recording a time.

`__exit__` receives the exception (or three `None`s if there wasn't one). Its return value decides what happens next: a truthy value means "I've handled it" and the exception disappears. Returning `False` (or `None`) lets it continue. Accidentally returning `True` would hide every error inside a timed block, a nasty bug.

`time.perf_counter()` is the clock to use for durations: it has the highest resolution available and never jumps backwards. `time.time()` is wall-clock time, which can jump when the system clock is adjusted.

`contextlib.contextmanager` offers a shorter way to write simple context managers with a generator and `try`/`finally`. The class version is clearer when the object needs to keep data, like `elapsed`, after the block.

### Exercise 6: `retry`

In [12]:
import functools
import time


def retry(times: int = 3, exceptions=(Exception,), delay: float = 0.0):
    if times < 1:
        raise ValueError(f"times must be at least 1, got {times}")

    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == times:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

In [13]:
check_retry(retry)

PASS  all 8 tests passed for retry


A decorator **factory** has three levels: `retry(...)` takes the settings and returns a decorator; the decorator takes the function and returns a wrapper; the wrapper runs on every call. Module 4 built single-level decorators; this adds the settings layer.

`except exceptions:` works with a tuple of exception classes, and only catches those. A `KeyError` from a bug in the function isn't something that retrying will fix, so it should surface immediately.

A bare `raise` inside an `except` block re-raises the exception being handled, with its original traceback, so the error message points at the real problem.

The checks count the calls to `time.sleep` (by temporarily replacing it), so they can tell a sleep after the last attempt from one between attempts, without actually waiting. Real retry code usually uses **exponential backoff**, doubling the delay after each failure, so a struggling server gets more time to recover.

### Exercise 7: `split_indices`

In [14]:
def split_indices(n: int, test_fraction: float, seed: int):
    if not 0 < test_fraction < 1:
        raise ValueError(f"test_fraction must be between 0 and 1, got {test_fraction}")
    n_test = round(n * test_fraction)
    if n_test == 0 or n_test == n:
        raise ValueError(f"a test fraction of {test_fraction} leaves an empty set with n = {n}")
    rng = np.random.default_rng(seed)
    order = rng.permutation(n)
    return np.sort(order[n_test:]), np.sort(order[:n_test])

In [15]:
check_split_indices(split_indices)

PASS  all 8 tests passed for split_indices


One permutation, then slice it: the first `n_test` positions are the test set and the rest are training. Because a permutation contains every index exactly once, the two parts can't overlap and nothing is lost. Drawing the two sets separately (for example with two calls to `rng.choice`) risks both.

The check on global state compares `np.random.get_state()` before and after the call. Code that calls `np.random.seed(seed)` "to be reproducible" passes the reproducibility test but changes the random numbers every **other** piece of code gets afterwards, which is the problem from lesson section 8.

Sorting the indices isn't required for correctness, but it keeps rows in their original order within each set, which makes results easier to inspect and compare.

`round` in Python 3 rounds halves to the nearest even number (`round(2.5)` is 2), which is fine here, but worth knowing if you ever see a test set that's one smaller than you expected.

### Exercise 8: `test_standardize`

In [16]:
import pytest


def test_standardize(standardize):
    rng = np.random.default_rng(0)
    X = rng.normal(loc=[5.0, -3.0, 100.0], scale=[1.0, 10.0, 0.5], size=(50, 3))
    before = X.copy()
    Z = standardize(X)

    # 1. per-column mean 0 and population std 1
    assert Z.shape == X.shape
    assert Z.mean(axis=0) == pytest.approx(0, abs=1e-12)
    assert Z.std(axis=0) == pytest.approx(1)

    # 3. the input is unchanged
    assert np.array_equal(X, before)

    # 2. a constant column becomes zeros
    C = np.column_stack([np.arange(5.0), np.full(5, 7.0)])
    Zc = standardize(C)
    assert np.all(np.isfinite(Zc))
    assert Zc[:, 1] == pytest.approx(0)

    # 4. only 2D input is accepted
    for bad in [np.arange(5.0), np.ones((2, 2, 2))]:
        with pytest.raises(ValueError):
            standardize(bad)

In [17]:
check_test_standardize(test_standardize)

PASS  all 6 tests passed for test_standardize


This exercise turns the usual setup around. Instead of tests checking your code, your tests are checked against code with known bugs. That's called **mutation testing**, and it answers the question "would my tests notice if something broke?"

The five broken versions each violate one line of the specification:

- **Global statistics**: one mean and standard deviation for the whole array. Caught only if the columns are on different scales, which is why the test data uses very different `loc` and `scale` values. With three similar columns, this bug would pass.
- **`ddof=1`**: standard deviations come out as `sqrt((n - 1) / n)`, about 0.99 with 50 rows. `pytest.approx`'s default tolerance (one part in a million) catches it; a loose tolerance like `abs=0.05` wouldn't.
- **Modifies the input**: caught by comparing with a copy taken **before** the call.
- **Divides by zero on a constant column**: caught by the constant-column case.
- **Accepts 1D input**: caught by the error check.

The lesson: tests need inputs that can tell a right answer from a nearly right one. Data where everything is on the same scale, or a tolerance that's too generous, lets bugs through while every test stays green.

### Exercise 9: `predict_in_batches`

In [18]:
def predict_in_batches(predict, X, batch_size: int):
    if batch_size < 1:
        raise ValueError(f"batch_size must be at least 1, got {batch_size}")
    outputs = [predict(X[start:start + batch_size]) for start in range(0, len(X), batch_size)]
    return np.concatenate(outputs, axis=0)

In [19]:
check_predict_in_batches(predict_in_batches)

PASS  all 6 tests passed for predict_in_batches


`range(0, len(X), batch_size)` gives the start of each batch: 0, `batch_size`, `2 * batch_size`, and so on. Slicing past the end of an array is allowed in Python and just returns the rows that exist, so the last, smaller batch needs no special case.

`np.concatenate(..., axis=0)` joins the pieces along the rows, which works for both 1D outputs (labels) and 2D outputs (class probabilities). `np.vstack` would turn a list of 1D arrays into a 2D one, a subtle bug the 1D check catches.

Memory now depends on `batch_size`, not on the size of `X`. The same pattern appears everywhere large data is processed: PyTorch's `DataLoader`, pandas' `chunksize` argument for reading big CSV files, and every training loop that works in mini-batches.

## Hard

### Exercise 10: `pairwise_distances`

In [20]:
def pairwise_distances(X, Y):
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    sq_x = np.sum(X ** 2, axis=1)[:, None]          # shape (n, 1)
    sq_y = np.sum(Y ** 2, axis=1)[None, :]          # shape (1, m)
    sq_dist = sq_x + sq_y - 2 * (X @ Y.T)           # shape (n, m)
    np.maximum(sq_dist, 0, out=sq_dist)             # rounding can give tiny negatives
    return np.sqrt(sq_dist)

In [21]:
check_pairwise_distances(pairwise_distances)

PASS  all 5 tests passed for pairwise_distances


Expanding the square replaces the `(n, m, d)` array of differences with three pieces: two vectors of squared lengths and one `(n, m)` matrix product. `X @ Y.T` runs in optimised linear algebra code (BLAS), which is also why this version is usually **faster** than broadcasting, not just smaller. It's how scikit-learn and most libraries compute distances.

The price is accuracy. When two points are very close compared with their distance from the origin, `‖x‖² + ‖y‖²` and `2 x·y` are large and nearly equal, and subtracting them is the catastrophic cancellation from Module 9. That's where the tiny negative values come from, and why the distance from a point to itself can come out as `1e-7` instead of exactly 0. The check uses a tolerance for that reason.

`np.maximum(sq_dist, 0, out=sq_dist)` clips in place, so it doesn't allocate a second `(n, m)` array. `np.clip` without `out=` would also be correct, just with a bit more memory.

For truly huge `n` and `m`, even the `(n, m)` output doesn't fit, and you'd combine this with `predict_in_batches` from the previous exercise, or use an approximate nearest-neighbour library.

### Exercise 11: `LRUCache`

In [22]:
from collections import OrderedDict


class LRUCache:
    def __init__(self, capacity: int):
        if capacity < 1:
            raise ValueError(f"capacity must be at least 1, got {capacity}")
        self.capacity = capacity
        self._items = OrderedDict()           # oldest use first, most recent use last
        self.hits = 0
        self.misses = 0

    def get(self, key, default=None):
        if key not in self._items:
            self.misses += 1
            return default
        self.hits += 1
        self._items.move_to_end(key)
        return self._items[key]

    def put(self, key, value) -> None:
        if key in self._items:
            self._items.move_to_end(key)
        self._items[key] = value
        if len(self._items) > self.capacity:
            self._items.popitem(last=False)   # remove the least recently used

    def __len__(self) -> int:
        return len(self._items)

    def __contains__(self, key) -> bool:
        return key in self._items

In [23]:
check_LRUCache(LRUCache)

PASS  all 7 tests passed for LRUCache


The `OrderedDict` keeps keys in order of last use: every `get` or `put` moves the key to the end, so the first key is always the least recently used, and `popitem(last=False)` removes it. Both operations are fast (constant time), however big the cache is.

The order of operations in `put` matters. Updating an existing key must move it to the end **before** the size check, otherwise a full cache could evict the key you just refreshed. And evicting after inserting, rather than before, handles both the "new key" and "existing key" cases with one check.

`get` uses `key not in self._items` rather than `self._items.get(key, default)`, because a stored value could legitimately be `None` or equal to `default`, and that must still count as a hit.

This is what `functools.lru_cache` does internally for function results. Caches like this sit in front of anything slow: loading images from disk during training, calling an embedding model for text you've already embedded, or looking up features for a user.

### Exercise 12: `save_run`

In [24]:
import json
import platform
from datetime import datetime, timezone


def _to_json(value):
    """Used by json.dumps for objects it can't handle on its own."""
    if isinstance(value, np.generic):         # np.float64, np.int64, np.bool_, ...
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f"cannot save a {type(value).__name__} in run.json")


def save_run(config: dict, metrics: dict, out_dir, overwrite: bool = False) -> Path:
    out_dir = Path(out_dir)
    path = out_dir / "run.json"
    if path.exists() and not overwrite:
        raise FileExistsError(f"{path} already exists; pass overwrite=True to replace it")

    record = {
        "config": config,
        "metrics": metrics,
        "environment": {"python": platform.python_version(), "numpy": np.__version__},
        "created": datetime.now(timezone.utc).isoformat(),
    }
    text = json.dumps(record, indent=2, default=_to_json)    # fails here, before any file is touched

    out_dir.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(text)
    temporary.replace(path)                                  # swap in the complete file in one step
    return path

In [25]:
check_save_run(save_run)

PASS  all 7 tests passed for save_run


**The `default=` argument of `json.dumps`** is called for any object the encoder doesn't understand. Returning a converted value lets the encoder carry on; raising `TypeError` reports the problem. `np.generic` is the parent class of every NumPy scalar type, and `.item()` converts any of them to the matching Python type.

**Never leave a broken file.** If the file were opened first and `json.dump` failed halfway, there'd be a truncated `run.json` on disk, and anything that later reads it would fail, or worse, the "already exists" check would block the retry. Building the text first means nothing touches the disk unless the data is valid.

**The temporary file and `replace`** protect against a different failure: the process being killed (out of memory, a timeout, a power cut) in the middle of writing. `Path.replace` swaps the finished temporary file into place in a single step on the same file system, so `run.json` is always either the old complete version or the new complete version.

**Refusing to overwrite** is cheap insurance. Re-running a cell with the same output folder is an easy mistake, and an hour-long experiment's results shouldn't vanish because of it.

## What's next

The mini project in `04_mini_project.ipynb` turns Module 9's penguin classifier from notebook code into a proper project: a package, a test suite, a config file, logging, a command-line interface and saved, traceable results.